# CSE440 NLP Project Notebook
## Multi-Class Text Classification: Outline, EDA, Preprocessing, and Experiment Pipeline

This notebook is structured to match the project brief:
- extensive EDA
- three preprocessing variants
- TF-IDF with one ML model and one deep neural network
- Skip-gram with all required sequence models
- accuracy, macro F1, confusion matrix, and classification report
- comparison of best and worst settings

This version is built from the files available in the workspace:
- `Training_data_4.csv`
- `Test_data.csv`
- `Spring 2026 - CSE440 Lab Project.pdf`

I did **not** find the actual CSE440 lab1/lab2/lab3 NLP reference files in the current workspace. I only found the project PDF plus the train/test CSVs, so the notebook below is aligned to the project requirements and the dataset itself.

## Recommended workflow

1. Inspect the raw train/test files and verify shapes, labels, missing values, and artifacts.
2. Do EDA on the training set first, then compare train vs test distributions only at a high level.
3. Build three text versions:
   - no preprocessing
   - extreme preprocessing
   - optimum preprocessing chosen from EDA
4. Keep the test set untouched until final evaluation.
5. Create a validation split from the training set only.
6. Run TF-IDF experiments with:
   - one ML model
   - one deep neural network
7. Run Skip-gram experiments with:
   - SimpleRNN
   - GRU
   - LSTM
   - Bidirectional SimpleRNN
   - Bidirectional GRU
   - Bidirectional LSTM
8. Log every tuning run in a results table.
9. Retrain the best validation setting on the full training set.
10. Evaluate once on the test set and write the report.

## Suggested group workflow

- Person 1: EDA + preprocessing analysis + report figures
- Person 2: TF-IDF + ML + DNN tuning
- Person 3: Skip-gram + sequence model tuning
- Everyone: final comparison table, report writing, and viva preparation

## Optional setup cell

In [ ]:
%pip install -q beautifulsoup4 wordcloud gensim nltk tensorflow

## Imports and reproducibility

In [ ]:
import os
import re
import html
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nltk
import tensorflow as tf

from bs4 import BeautifulSoup
from collections import Counter
from wordcloud import WordCloud
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import TruncatedSVD

from nltk.stem import PorterStemmer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Embedding, SimpleRNN, GRU, LSTM, Bidirectional, SpatialDropout1D

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.max_columns", 100)

nltk.download("punkt")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

os.makedirs("figures", exist_ok=True)
os.makedirs("logs", exist_ok=True)

## Load the assigned train and test files

In [ ]:
TRAIN_PATH = "Training_data_4.csv"
TEST_PATH = "Test_data.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print(train_df.shape)
print(test_df.shape)
train_df.head()

## Dataset overview

In [ ]:
overview_df = pd.DataFrame({
    "split": ["train", "test"],
    "rows": [len(train_df), len(test_df)],
    "missing_text": [train_df["News Headline"].isna().sum(), test_df["News Headline"].isna().sum()],
    "missing_label": [train_df["News Topic"].isna().sum(), test_df["News Topic"].isna().sum()],
    "unique_raw_texts": [train_df["News Headline"].nunique(), test_df["News Headline"].nunique()],
    "duplicate_rows": [train_df.duplicated().sum(), test_df.duplicated().sum()]
})
overview_df

## Class distribution table

In [ ]:
train_label_dist = train_df["News Topic"].value_counts().rename("train_count").to_frame()
train_label_dist["train_pct"] = (train_label_dist["train_count"] / len(train_df) * 100).round(2)

test_label_dist = test_df["News Topic"].value_counts().rename("test_count").to_frame()
test_label_dist["test_pct"] = (test_label_dist["test_count"] / len(test_df) * 100).round(2)

label_dist_df = train_label_dist.join(test_label_dist, how="outer")
label_dist_df

## Class distribution plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = pd.DataFrame({
    "Train": train_df["News Topic"].value_counts(),
    "Test": test_df["News Topic"].value_counts()
}).fillna(0)

plot_df.plot(kind="bar", ax=ax)
ax.set_title("Class distribution in train vs test")
ax.set_xlabel("News Topic")
ax.set_ylabel("Count")
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## Raw sample inspection

In [ ]:
sample_view = train_df.sample(5, random_state=SEED)[["News Topic", "News Headline"]]
sample_view

## Cleaning helpers for EDA

In [ ]:
ENTITY_REPLACEMENTS = {
    r"#39;": "'",
    r"#146;": "'",
    r"quot;": '"',
    r"amp;": "&",
    r"#151;": "-",
    r"#36;": "$",
    r"&lt;": "<",
    r"&gt;": ">"
}

def normalize_artifacts(text):
    text = str(text)
    for pattern, replacement in ENTITY_REPLACEMENTS.items():
        text = re.sub(pattern, replacement, text)
    text = text.replace("\\", " ")
    return text

def basic_eda_clean(text):
    text = normalize_artifacts(text)
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    text = html.unescape(text)
    text = re.sub(r"^\s*News Headlines:\s*", "", text, flags=re.I)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [train_df, test_df]:
    df["text_eda"] = df["News Headline"].apply(basic_eda_clean)
    df["char_len"] = df["text_eda"].str.len()
    df["word_len"] = df["text_eda"].str.count(r"\b\w+\b")
    df["has_digit"] = df["text_eda"].str.contains(r"\d")
    df["has_parentheses"] = df["text_eda"].str.contains(r"[()]")
    df["has_hyphen"] = df["text_eda"].str.contains(r"-")
    df["has_quote"] = df["text_eda"].str.contains(r"['\"]")
    df["has_backslash"] = df["News Headline"].str.contains(r"\\")
    df["has_html_tags"] = df["News Headline"].str.contains(r"<html>|<body>|<br>|<b>", regex=True)
    df["has_broken_entities"] = df["News Headline"].str.contains(r"#39;|quot;|amp;|#151;|#36;|&lt;|&gt;", regex=True)
    df["starts_with_news_headlines"] = df["News Headline"].str.contains("News Headlines:", regex=False)

train_df[["News Topic", "text_eda"]].head()

## Data-quality summary

In [ ]:
quality_df = pd.DataFrame({
    "metric": [
        "rows",
        "missing_text",
        "missing_label",
        "unique_raw_texts",
        "duplicate_rows",
        "avg_words",
        "median_words",
        "avg_chars",
        "median_chars",
        "has_html_tags_pct",
        "has_broken_entities_pct",
        "has_backslash_pct",
        "starts_with_news_headlines_pct"
    ],
    "train": [
        len(train_df),
        train_df["News Headline"].isna().sum(),
        train_df["News Topic"].isna().sum(),
        train_df["News Headline"].nunique(),
        train_df.duplicated().sum(),
        round(train_df["word_len"].mean(), 2),
        int(train_df["word_len"].median()),
        round(train_df["char_len"].mean(), 2),
        int(train_df["char_len"].median()),
        round(train_df["has_html_tags"].mean() * 100, 2),
        round(train_df["has_broken_entities"].mean() * 100, 2),
        round(train_df["has_backslash"].mean() * 100, 2),
        round(train_df["starts_with_news_headlines"].mean() * 100, 2)
    ],
    "test": [
        len(test_df),
        test_df["News Headline"].isna().sum(),
        test_df["News Topic"].isna().sum(),
        test_df["News Headline"].nunique(),
        test_df.duplicated().sum(),
        round(test_df["word_len"].mean(), 2),
        int(test_df["word_len"].median()),
        round(test_df["char_len"].mean(), 2),
        int(test_df["char_len"].median()),
        round(test_df["has_html_tags"].mean() * 100, 2),
        round(test_df["has_broken_entities"].mean() * 100, 2),
        round(test_df["has_backslash"].mean() * 100, 2),
        round(test_df["starts_with_news_headlines"].mean() * 100, 2)
    ]
})
quality_df

## Duplicate analysis

In [ ]:
dup_df = train_df.assign(dup_any=train_df["News Headline"].duplicated(keep=False)).groupby("News Topic").agg(
    total=("News Headline", "size"),
    dup_rows=("dup_any", "sum"),
    repeated_beyond_first=("News Headline", lambda s: int(s.duplicated().sum()))
).reset_index()

dup_df["dup_row_pct"] = (dup_df["dup_rows"] / dup_df["total"] * 100).round(2)
dup_df["unique_texts"] = dup_df["total"] - dup_df["repeated_beyond_first"]
dup_df

## Length and class-wise statistics

In [ ]:
length_stats_df = train_df.groupby("News Topic").agg(
    count=("text_eda", "size"),
    avg_words=("word_len", "mean"),
    median_words=("word_len", "median"),
    avg_chars=("char_len", "mean"),
    median_chars=("char_len", "median"),
    digits_pct=("has_digit", "mean"),
    parentheses_pct=("has_parentheses", "mean"),
    hyphen_pct=("has_hyphen", "mean"),
    quote_pct=("has_quote", "mean"),
    broken_entities_pct=("has_broken_entities", "mean")
).reset_index()

for col in length_stats_df.columns:
    if col.endswith("_pct"):
        length_stats_df[col] = (length_stats_df[col] * 100).round(2)
    elif col not in ["News Topic", "count"]:
        length_stats_df[col] = length_stats_df[col].round(2)

length_stats_df

## Word-count distribution

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(train_df["word_len"], bins=50, alpha=0.7, label="Train")
ax.hist(test_df["word_len"], bins=50, alpha=0.5, label="Test")
ax.set_title("Word-count distribution")
ax.set_xlabel("Word count")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## Word-count by class

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ordered_topics = sorted(train_df["News Topic"].unique())
box_data = [train_df.loc[train_df["News Topic"] == topic, "word_len"] for topic in ordered_topics]
ax.boxplot(box_data, labels=ordered_topics, showfliers=False)
ax.set_title("Word-count distribution by class")
ax.set_ylabel("Word count")
plt.xticks(rotation=20)
ax.grid(axis="y", linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()

## Source-tag frequency by class

In [ ]:
source_df = []
for topic, subset in train_df.groupby("News Topic"):
    source_df.append({
        "News Topic": topic,
        "Reuters_pct": round(subset["text_eda"].str.contains(r"\bReuters\b").mean() * 100, 2),
        "AP_pct": round(subset["text_eda"].str.contains(r"\bAP\b").mean() * 100, 2),
        "AFP_pct": round(subset["text_eda"].str.contains(r"\bAFP\b").mean() * 100, 2)
    })

source_df = pd.DataFrame(source_df)
source_df

## Most common unigrams after removing source tags

In [ ]:
def remove_source_tags(text):
    text = re.sub(r"\((Reuters|AP|AFP)\)", " ", text)
    text = re.sub(r"\b(?:Reuters|AP|AFP)\b\s*[-:)]?", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_df["text_no_source"] = train_df["text_eda"].apply(remove_source_tags)

def get_top_ngrams(texts, ngram_range=(1, 1), top_n=15):
    vectorizer = CountVectorizer(lowercase=True, stop_words="english", ngram_range=ngram_range, min_df=5)
    matrix = vectorizer.fit_transform(texts)
    counts = np.asarray(matrix.sum(axis=0)).ravel()
    terms = np.array(vectorizer.get_feature_names_out())
    order = counts.argsort()[::-1][:top_n]
    return pd.DataFrame({"term": terms[order], "count": counts[order]})

top_unigrams = {topic: get_top_ngrams(group["text_no_source"], (1, 1), 12) for topic, group in train_df.groupby("News Topic")}
top_bigrams = {topic: get_top_ngrams(group["text_no_source"], (2, 2), 12) for topic, group in train_df.groupby("News Topic")}

top_unigrams["Science and Technology"]

## Most common bigrams after removing source tags

In [ ]:
top_bigrams["Sports"]

## Word clouds by class

In [ ]:
topic_list = sorted(train_df["News Topic"].unique())

for topic in topic_list:
    text_blob = " ".join(train_df.loc[train_df["News Topic"] == topic, "text_no_source"])
    cloud = WordCloud(width=1200, height=600, background_color="white", collocations=False).generate(text_blob)

    plt.figure(figsize=(12, 6))
    plt.imshow(cloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word cloud: {topic}")
    plt.tight_layout()
    plt.show()

## EDA findings already visible from the dataset

1. The dataset has **104,763 training rows** and **12,000 test rows**, with **4 classes** and **no missing text or label values**.
2. The **training set is imbalanced**:
   - Science and Technology: **34.77%**
   - Sports: **32.62%**
   - Business: **17.72%**
   - World News: **14.89%**
   The **test set is perfectly balanced at 25% per class**, so macro F1 and stratified validation matter a lot.
3. Every row contains **HTML wrappers and the constant prefix `News Headlines:`**. That means HTML stripping is not optional for the preprocessed versions.
4. About **32.16%** of training rows contain broken HTML-style entities such as `#39;`, `quot;`, `amp;`, or `#151;`.
5. About **10.48%** of training rows contain backslash artifacts like `\said` or `\to`.
6. The texts are **not just short headlines**. After basic cleanup, the training texts average about **38.38 words** and **229.42 characters**, which means they behave more like **headline + short news snippet**.
7. The training set has **31,208 exact duplicate rows**, while the test set has **0**. Duplicates are concentrated almost entirely in:
   - Science and Technology: **74.15%** of rows are part of duplicate groups
   - Sports: **71.81%**
   Business and World News have effectively **0% exact duplicate rows**.
8. Digits are common in the text, especially for Sports and Business. Removing numbers in the optimum pipeline is probably a bad idea.
9. Source-wire tags are common and differ by class:
   - Business has strong Reuters presence (**18.90%**)
   - Sports has stronger AP presence (**10.35%**)
   - World News uses a mixture of Reuters, AP, and AFP
   This means source markers may create **publisher-style bias**, so removing them is a good candidate for the optimum preprocessing pipeline.
10. Exact raw train/test overlap is **0**, and overlap after basic cleanup is still very small (**15 texts**), so leakage does not look like the main issue.

## Recommended optimum preprocessing from this EDA

For this dataset, the optimum version should usually:
- remove HTML tags and the `News Headlines:` boilerplate
- normalize broken entities and backslash artifacts
- lowercase the text
- normalize whitespace
- remove source tags like Reuters/AP/AFP
- keep numbers
- keep most stopwords
- avoid stemming

The extreme version can still remove stopwords and apply stemming because the project explicitly asks for it, but the EDA suggests that **extreme preprocessing will probably lose useful meaning** for this dataset.

## Preprocessing design

We will create exactly three text versions:

1. No preprocessing
   - uses the raw text exactly as provided

2. Extreme preprocessing
   - HTML removal
   - broken entity normalization
   - source-tag removal
   - lowercasing
   - non-letter removal
   - stopword removal
   - stemming

3. Optimum preprocessing
   - HTML removal
   - broken entity normalization
   - source-tag removal
   - lowercasing
   - whitespace normalization
   - keeps numbers
   - keeps most stopwords
   - no stemming

The optimum version follows the EDA findings.

## Build the three preprocessing variants

In [ ]:
STOP_WORDS = set(ENGLISH_STOP_WORDS)
STEMMER = PorterStemmer()

def preprocess_none(text):
    return str(text)

def preprocess_extreme(text):
    text = basic_eda_clean(text)
    text = re.sub(r"\((Reuters|AP|AFP)\)", " ", text)
    text = re.sub(r"\b(?:Reuters|AP|AFP)\b\s*[-:)]?", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z'\s]", " ", text)
    tokens = re.findall(r"[a-z']+", text)
    tokens = [token for token in tokens if token not in STOP_WORDS and len(token) > 1]
    tokens = [STEMMER.stem(token) for token in tokens]
    return " ".join(tokens)

def preprocess_optimum(text):
    text = basic_eda_clean(text)
    text = re.sub(r"\((Reuters|AP|AFP)\)", " ", text)
    text = re.sub(r"\b(?:Reuters|AP|AFP)\b\s*[-:)]?", " ", text)
    text = text.lower()
    text = re.sub(r"[^a-z0-9'\-\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train_versions = {
    "none": train_df["News Headline"].apply(preprocess_none),
    "extreme": train_df["News Headline"].apply(preprocess_extreme),
    "optimum": train_df["News Headline"].apply(preprocess_optimum)
}

test_versions = {
    "none": test_df["News Headline"].apply(preprocess_none),
    "extreme": test_df["News Headline"].apply(preprocess_extreme),
    "optimum": test_df["News Headline"].apply(preprocess_optimum)
}

## Preview the three versions side by side

In [ ]:
preview_df = pd.DataFrame({
    "raw": train_df["News Headline"].head(3),
    "no_preprocessing": train_versions["none"].head(3),
    "extreme": train_versions["extreme"].head(3),
    "optimum": train_versions["optimum"].head(3)
})
preview_df

## Optional extra experiment: deduplicate the training set

In [ ]:
train_df_dedup = train_df.drop_duplicates(subset=["News Headline"]).copy()
print(train_df.shape)
print(train_df_dedup.shape)

train_versions_dedup = {
    "none": train_df_dedup["News Headline"].apply(preprocess_none),
    "extreme": train_df_dedup["News Headline"].apply(preprocess_extreme),
    "optimum": train_df_dedup["News Headline"].apply(preprocess_optimum)
}

## Validation split, label encoding, and class weights

In [ ]:
TARGET_COLUMN = "News Topic"
TEXT_COLUMN = "text"

label_encoder = LabelEncoder()
label_encoder.fit(train_df[TARGET_COLUMN])

def make_split(text_series, label_series, test_size=0.15):
    split_df = pd.DataFrame({TEXT_COLUMN: text_series, TARGET_COLUMN: label_series})
    X_train, X_val, y_train, y_val = train_test_split(
        split_df[TEXT_COLUMN],
        split_df[TARGET_COLUMN],
        test_size=test_size,
        random_state=SEED,
        stratify=split_df[TARGET_COLUMN]
    )
    y_train_enc = label_encoder.transform(y_train)
    y_val_enc = label_encoder.transform(y_val)

    class_weights_array = compute_class_weight(
        class_weight="balanced",
        classes=np.unique(y_train_enc),
        y=y_train_enc
    )
    class_weight_dict = {i: class_weights_array[i] for i in range(len(class_weights_array))}
    return X_train, X_val, y_train, y_val, y_train_enc, y_val_enc, class_weight_dict

X_train_opt, X_val_opt, y_train_opt, y_val_opt, y_train_opt_enc, y_val_opt_enc, class_weights_opt = make_split(
    train_versions["optimum"],
    train_df[TARGET_COLUMN]
)

print(X_train_opt.shape, X_val_opt.shape)
print(class_weights_opt)

## Evaluation helpers

In [ ]:
EXPERIMENT_LOG = []

def evaluate_predictions(y_true, y_pred, labels):
    report_dict = classification_report(y_true, y_pred, output_dict=True)
    report_df = pd.DataFrame(report_dict).T
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro")
    }
    return metrics, report_df

def show_confusion(y_true, y_pred, labels, title):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    fig, ax = plt.subplots(figsize=(7, 7))
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

def log_result(preprocessing_name, representation, model_name, split_name, metrics, params):
    EXPERIMENT_LOG.append({
        "preprocessing": preprocessing_name,
        "representation": representation,
        "model": model_name,
        "split": split_name,
        "accuracy": round(metrics["accuracy"], 4),
        "macro_f1": round(metrics["macro_f1"], 4),
        "params": params
    })

## Tuning log helper

In [ ]:
TUNING_LOG = []

def log_tuning(stage, preprocessing_name, model_name, params, val_accuracy, val_macro_f1, notes=""):
    TUNING_LOG.append({
        "stage": stage,
        "preprocessing": preprocessing_name,
        "model": model_name,
        "params": str(params),
        "val_accuracy": round(val_accuracy, 4),
        "val_macro_f1": round(val_macro_f1, 4),
        "notes": notes
    })

def tuning_table():
    return pd.DataFrame(TUNING_LOG).sort_values(["stage", "val_macro_f1"], ascending=[True, False])

## TF-IDF feature helper

In [ ]:
def build_tfidf_features(X_train, X_val, max_features=50000, ngram_range=(1, 2), min_df=3, max_df=0.95):
    vectorizer = TfidfVectorizer(
        max_features=max_features,
        ngram_range=ngram_range,
        min_df=min_df,
        max_df=max_df,
        sublinear_tf=True
    )
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_val_tfidf = vectorizer.transform(X_val)
    return vectorizer, X_train_tfidf, X_val_tfidf

## TF-IDF with Logistic Regression

In [ ]:
def run_logistic_regression(preprocessing_name, train_text, train_labels, class_weight="balanced", C=3.0):
    X_train, X_val, y_train, y_val, y_train_enc, y_val_enc, _ = make_split(train_text, train_labels)
    vectorizer, X_train_tfidf, X_val_tfidf = build_tfidf_features(
        X_train,
        X_val,
        max_features=40000,
        ngram_range=(1, 2),
        min_df=3
    )

    model = LogisticRegression(
        C=C,
        max_iter=1500,
        class_weight=class_weight
    )
    model.fit(X_train_tfidf, y_train)
    val_pred = model.predict(X_val_tfidf)

    metrics, report_df = evaluate_predictions(y_val, val_pred, label_encoder.classes_)
    log_result(preprocessing_name, "TF-IDF", "LogisticRegression", "validation", metrics, {"C": C, "class_weight": class_weight})
    log_tuning("TF-IDF_ML", preprocessing_name, "LogisticRegression", {"C": C, "class_weight": class_weight}, metrics["accuracy"], metrics["macro_f1"])

    print(metrics)
    display(report_df)
    show_confusion(y_val, val_pred, label_encoder.classes_, f"Logistic Regression | {preprocessing_name}")
    return model, vectorizer, metrics, report_df

## Run Logistic Regression on all three preprocessing variants

In [ ]:
logreg_runs = {}

for preprocessing_name in ["none", "extreme", "optimum"]:
    print("=" * 80)
    print(f"Running Logistic Regression for: {preprocessing_name}")
    logreg_runs[preprocessing_name] = run_logistic_regression(
        preprocessing_name,
        train_versions[preprocessing_name],
        train_df["News Topic"],
        class_weight="balanced",
        C=3.0
    )

## Optional TF-IDF baselines: MultinomialNB and RandomForest

In [ ]:
def run_multinomial_nb(preprocessing_name, train_text, train_labels, alpha=0.5):
    X_train, X_val, y_train, y_val, _, _, _ = make_split(train_text, train_labels)
    vectorizer, X_train_tfidf, X_val_tfidf = build_tfidf_features(
        X_train,
        X_val,
        max_features=40000,
        ngram_range=(1, 2),
        min_df=3
    )

    model = MultinomialNB(alpha=alpha)
    model.fit(X_train_tfidf, y_train)
    val_pred = model.predict(X_val_tfidf)

    metrics, report_df = evaluate_predictions(y_val, val_pred, label_encoder.classes_)
    log_result(preprocessing_name, "TF-IDF", "MultinomialNB", "validation", metrics, {"alpha": alpha})
    log_tuning("TF-IDF_ML", preprocessing_name, "MultinomialNB", {"alpha": alpha}, metrics["accuracy"], metrics["macro_f1"])
    return model, vectorizer, metrics, report_df

def run_random_forest(preprocessing_name, train_text, train_labels, n_estimators=300, max_depth=None):
    X_train, X_val, y_train, y_val, _, _, _ = make_split(train_text, train_labels)
    vectorizer, X_train_tfidf, X_val_tfidf = build_tfidf_features(
        X_train,
        X_val,
        max_features=20000,
        ngram_range=(1, 2),
        min_df=3
    )

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=SEED,
        n_jobs=-1
    )
    model.fit(X_train_tfidf, y_train)
    val_pred = model.predict(X_val_tfidf)

    metrics, report_df = evaluate_predictions(y_val, val_pred, label_encoder.classes_)
    log_result(preprocessing_name, "TF-IDF", "RandomForest", "validation", metrics, {"n_estimators": n_estimators, "max_depth": max_depth})
    log_tuning("TF-IDF_ML", preprocessing_name, "RandomForest", {"n_estimators": n_estimators, "max_depth": max_depth}, metrics["accuracy"], metrics["macro_f1"])
    return model, vectorizer, metrics, report_df

## TF-IDF + Deep Neural Network

The PDF asks for a DNN with at least 3 to 4+ layers and 128+ neurons in the first layer.

Because raw TF-IDF can be very high-dimensional, two practical options are:

1. use sparse TF-IDF directly as Keras input
2. reduce TF-IDF with TruncatedSVD first, then train the DNN on dense vectors

The cells below implement the safer dense version using TruncatedSVD. It still starts from TF-IDF.

## DNN builder and training function

In [ ]:
def build_tfidf_dnn(input_dim, n_classes, learning_rate=1e-3, dropout_rate=0.3):
    model = Sequential([
        Dense(512, activation="relu", input_shape=(input_dim,)),
        Dropout(dropout_rate),
        Dense(256, activation="relu"),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def run_tfidf_dnn(preprocessing_name, train_text, train_labels, n_components=1024, batch_size=256, epochs=15):
    X_train, X_val, y_train, y_val, y_train_enc, y_val_enc, class_weight_dict = make_split(train_text, train_labels)
    vectorizer, X_train_tfidf, X_val_tfidf = build_tfidf_features(
        X_train,
        X_val,
        max_features=50000,
        ngram_range=(1, 2),
        min_df=3
    )

    svd = TruncatedSVD(n_components=n_components, random_state=SEED)
    X_train_dense = svd.fit_transform(X_train_tfidf).astype("float32")
    X_val_dense = svd.transform(X_val_tfidf).astype("float32")

    model = build_tfidf_dnn(X_train_dense.shape[1], len(label_encoder.classes_))

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
    ]

    history = model.fit(
        X_train_dense,
        y_train_enc,
        validation_data=(X_val_dense, y_val_enc),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=callbacks,
        verbose=1
    )

    val_pred = model.predict(X_val_dense, verbose=0).argmax(axis=1)
    val_pred_labels = label_encoder.inverse_transform(val_pred)

    metrics, report_df = evaluate_predictions(y_val, val_pred_labels, label_encoder.classes_)
    log_result(preprocessing_name, "TF-IDF", "DeepNN", "validation", metrics, {"n_components": n_components, "batch_size": batch_size, "epochs": epochs})
    log_tuning("TF-IDF_DNN", preprocessing_name, "DeepNN", {"n_components": n_components, "batch_size": batch_size, "epochs": epochs}, metrics["accuracy"], metrics["macro_f1"])

    print(metrics)
    display(report_df)
    show_confusion(y_val, val_pred_labels, label_encoder.classes_, f"DNN | {preprocessing_name}")
    return model, vectorizer, svd, history, metrics, report_df

## Run the TF-IDF DNN on all three preprocessing variants

In [ ]:
dnn_runs = {}

for preprocessing_name in ["none", "extreme", "optimum"]:
    print("=" * 80)
    print(f"Running DNN for: {preprocessing_name}")
    dnn_runs[preprocessing_name] = run_tfidf_dnn(
        preprocessing_name,
        train_versions[preprocessing_name],
        train_df["News Topic"],
        n_components=1024,
        batch_size=256,
        epochs=12
    )

## Skip-gram preparation

The recurrent models in the PDF should use Skip-gram embeddings.

The workflow below is:
1. tokenize the chosen preprocessed text
2. train Word2Vec with `sg=1`
3. build a tokenizer and padded sequences
4. create the embedding matrix
5. train RNN / GRU / LSTM / bidirectional variants

## Skip-gram helpers

In [ ]:
def tokenize_for_skipgram(text_series):
    return [text.split() for text in text_series]

def train_skipgram_embeddings(tokenized_texts, vector_size=200, window=5, min_count=2, epochs=15):
    model = Word2Vec(
        sentences=tokenized_texts,
        vector_size=vector_size,
        window=window,
        min_count=min_count,
        sg=1,
        workers=4,
        epochs=epochs,
        seed=SEED
    )
    return model

def build_sequence_data(train_text, val_text, max_vocab=50000, max_len=60):
    tokenizer = Tokenizer(num_words=max_vocab, oov_token="<OOV>")
    tokenizer.fit_on_texts(train_text)

    X_train_seq = tokenizer.texts_to_sequences(train_text)
    X_val_seq = tokenizer.texts_to_sequences(val_text)

    X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding="post", truncating="post")
    X_val_pad = pad_sequences(X_val_seq, maxlen=max_len, padding="post", truncating="post")
    return tokenizer, X_train_pad, X_val_pad

def build_embedding_matrix(tokenizer, w2v_model, max_vocab=50000):
    embedding_dim = w2v_model.vector_size
    vocab_size = min(max_vocab, len(tokenizer.word_index) + 1)

    embedding_matrix = np.random.normal(0, 0.05, (vocab_size, embedding_dim))
    embedding_matrix[0] = np.zeros(embedding_dim)

    for word, idx in tokenizer.word_index.items():
        if idx >= vocab_size:
            continue
        if word in w2v_model.wv:
            embedding_matrix[idx] = w2v_model.wv[word]

    return embedding_matrix

## Prepare tokenizer, sequences, and embedding matrix

In [ ]:
def prepare_skipgram_inputs(preprocessing_name, train_text_series, train_labels, max_vocab=50000, max_len=60):
    X_train, X_val, y_train, y_val, y_train_enc, y_val_enc, class_weight_dict = make_split(train_text_series, train_labels)

    tokenized_train = tokenize_for_skipgram(X_train)
    w2v_model = train_skipgram_embeddings(
        tokenized_train,
        vector_size=200,
        window=5,
        min_count=2,
        epochs=15
    )

    tokenizer, X_train_pad, X_val_pad = build_sequence_data(X_train, X_val, max_vocab=max_vocab, max_len=max_len)
    embedding_matrix = build_embedding_matrix(tokenizer, w2v_model, max_vocab=max_vocab)

    return {
        "preprocessing": preprocessing_name,
        "X_train": X_train,
        "X_val": X_val,
        "y_train": y_train,
        "y_val": y_val,
        "y_train_enc": y_train_enc,
        "y_val_enc": y_val_enc,
        "class_weights": class_weight_dict,
        "tokenizer": tokenizer,
        "X_train_pad": X_train_pad,
        "X_val_pad": X_val_pad,
        "embedding_matrix": embedding_matrix,
        "max_vocab": max_vocab,
        "max_len": max_len
    }

## Recurrent model builders

In [ ]:
def build_simple_rnn_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        SimpleRNN(units),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

def build_gru_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        GRU(units),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

def build_lstm_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        LSTM(units),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

def build_bi_simple_rnn_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        Bidirectional(SimpleRNN(units)),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

def build_bi_gru_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        Bidirectional(GRU(units)),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

def build_bi_lstm_model(vocab_size, embedding_matrix, max_len, n_classes, units=128, dropout_rate=0.3):
    model = Sequential([
        Embedding(vocab_size, embedding_matrix.shape[1], weights=[embedding_matrix], input_length=max_len, trainable=False),
        SpatialDropout1D(dropout_rate),
        Bidirectional(LSTM(units)),
        Dropout(dropout_rate),
        Dense(128, activation="relu"),
        Dropout(dropout_rate),
        Dense(n_classes, activation="softmax")
    ])
    return model

## Unified training helper for sequence models

In [ ]:
def compile_model(model, learning_rate=1e-3):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def run_sequence_model(prepared_data, model_name, units=128, dropout_rate=0.3, batch_size=256, epochs=12):
    embedding_matrix = prepared_data["embedding_matrix"]
    vocab_size = embedding_matrix.shape[0]
    max_len = prepared_data["max_len"]
    n_classes = len(label_encoder.classes_)

    builders = {
        "SimpleRNN": build_simple_rnn_model,
        "GRU": build_gru_model,
        "LSTM": build_lstm_model,
        "BiSimpleRNN": build_bi_simple_rnn_model,
        "BiGRU": build_bi_gru_model,
        "BiLSTM": build_bi_lstm_model
    }

    model = builders[model_name](
        vocab_size=vocab_size,
        embedding_matrix=embedding_matrix,
        max_len=max_len,
        n_classes=n_classes,
        units=units,
        dropout_rate=dropout_rate
    )
    model = compile_model(model)

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)
    ]

    history = model.fit(
        prepared_data["X_train_pad"],
        prepared_data["y_train_enc"],
        validation_data=(prepared_data["X_val_pad"], prepared_data["y_val_enc"]),
        epochs=epochs,
        batch_size=batch_size,
        class_weight=prepared_data["class_weights"],
        callbacks=callbacks,
        verbose=1
    )

    val_pred = model.predict(prepared_data["X_val_pad"], verbose=0).argmax(axis=1)
    val_pred_labels = label_encoder.inverse_transform(val_pred)
    y_val_labels = prepared_data["y_val"]

    metrics, report_df = evaluate_predictions(y_val_labels, val_pred_labels, label_encoder.classes_)
    log_result(prepared_data["preprocessing"], "Skip-gram", model_name, "validation", metrics, {"units": units, "dropout_rate": dropout_rate, "batch_size": batch_size, "epochs": epochs})
    log_tuning("SKIPGRAM_RNN", prepared_data["preprocessing"], model_name, {"units": units, "dropout_rate": dropout_rate, "batch_size": batch_size, "epochs": epochs}, metrics["accuracy"], metrics["macro_f1"])

    print(metrics)
    display(report_df)
    show_confusion(y_val_labels, val_pred_labels, label_encoder.classes_, f"{model_name} | {prepared_data['preprocessing']}")
    return model, history, metrics, report_df

## Prepare one Skip-gram dataset first

In [ ]:
prepared_optimum = prepare_skipgram_inputs(
    "optimum",
    train_versions["optimum"],
    train_df["News Topic"],
    max_vocab=50000,
    max_len=60
)

prepared_optimum.keys()

## Run the six required sequence models on the optimum preprocessing first

In [ ]:
sequence_runs_optimum = {}

for model_name in ["SimpleRNN", "GRU", "LSTM", "BiSimpleRNN", "BiGRU", "BiLSTM"]:
    print("=" * 80)
    print(f"Running {model_name} on optimum preprocessing")
    sequence_runs_optimum[model_name] = run_sequence_model(
        prepared_optimum,
        model_name=model_name,
        units=128,
        dropout_rate=0.3,
        batch_size=256,
        epochs=12
    )

## Full long-running cell: run the six sequence models on all three preprocessing variants

In [ ]:
sequence_runs_all = {}

for preprocessing_name in ["none", "extreme", "optimum"]:
    prepared_data = prepare_skipgram_inputs(
        preprocessing_name,
        train_versions[preprocessing_name],
        train_df["News Topic"],
        max_vocab=50000,
        max_len=60
    )

    for model_name in ["SimpleRNN", "GRU", "LSTM", "BiSimpleRNN", "BiGRU", "BiLSTM"]:
        print("=" * 80)
        print(f"Running {model_name} on {preprocessing_name}")
        sequence_runs_all[(preprocessing_name, model_name)] = run_sequence_model(
            prepared_data,
            model_name=model_name,
            units=128,
            dropout_rate=0.3,
            batch_size=256,
            epochs=12
        )

## Experiment results table

In [ ]:
results_df = pd.DataFrame(EXPERIMENT_LOG)
results_df.sort_values(["representation", "macro_f1"], ascending=[True, False])

## Experiment comparison plot

In [ ]:
if len(EXPERIMENT_LOG) > 0:
    results_plot_df = pd.DataFrame(EXPERIMENT_LOG).sort_values("macro_f1", ascending=False)

    plt.figure(figsize=(12, 6))
    plt.bar(results_plot_df["model"] + " | " + results_plot_df["preprocessing"], results_plot_df["macro_f1"])
    plt.title("Validation macro F1 across experiments")
    plt.xlabel("Experiment")
    plt.ylabel("Macro F1")
    plt.xticks(rotation=75, ha="right")
    plt.grid(axis="y", linestyle="--", alpha=0.3)
    plt.tight_layout()
    plt.show()

    display(results_plot_df)

## Final test evaluation

Do this only after you finish manual tuning on the validation split.

The recommended process is:
1. choose the best configuration by validation macro F1
2. rebuild the chosen preprocessing on the full training set
3. retrain the chosen model on the full training set
4. transform the untouched test set once
5. report test accuracy, macro F1, confusion matrix, and classification report

## Final test evaluation helper

In [ ]:
def final_test_evaluation(best_preprocessing_name, best_vectorizer_params=None, best_model_params=None):
    if best_vectorizer_params is None:
        best_vectorizer_params = {"max_features": 40000, "ngram_range": (1, 2), "min_df": 3, "max_df": 0.95}
    if best_model_params is None:
        best_model_params = {"C": 3.0, "class_weight": "balanced"}

    X_train_full = train_versions[best_preprocessing_name]
    X_test_final = test_versions[best_preprocessing_name]
    y_train_full = train_df["News Topic"]
    y_test_final = test_df["News Topic"]

    vectorizer = TfidfVectorizer(
        max_features=best_vectorizer_params["max_features"],
        ngram_range=best_vectorizer_params["ngram_range"],
        min_df=best_vectorizer_params["min_df"],
        max_df=best_vectorizer_params["max_df"],
        sublinear_tf=True
    )

    X_train_full_tfidf = vectorizer.fit_transform(X_train_full)
    X_test_final_tfidf = vectorizer.transform(X_test_final)

    model = LogisticRegression(
        C=best_model_params["C"],
        max_iter=1500,
        class_weight=best_model_params["class_weight"]
    )
    model.fit(X_train_full_tfidf, y_train_full)
    test_pred = model.predict(X_test_final_tfidf)

    metrics, report_df = evaluate_predictions(y_test_final, test_pred, label_encoder.classes_)
    print(metrics)
    display(report_df)
    show_confusion(y_test_final, test_pred, label_encoder.classes_, f"Final test | Logistic Regression | {best_preprocessing_name}")
    return model, vectorizer, metrics, report_df

## Example final test evaluation cell

In [ ]:
final_model, final_vectorizer, final_metrics, final_report = final_test_evaluation(
    best_preprocessing_name="optimum",
    best_vectorizer_params={"max_features": 40000, "ngram_range": (1, 2), "min_df": 3, "max_df": 0.95},
    best_model_params={"C": 3.0, "class_weight": "balanced"}
)

## Report outline aligned to the PDF

1. Abstract
   - one paragraph with dataset, preprocessing variants, representations, models, and best result

2. Introduction
   - task definition
   - why multi-class news classification matters
   - short summary of the assigned dataset

3. Methodology
   - Dataset
   - EDA findings
   - Preprocessing
   - TF-IDF
   - Skip-gram
   - Model architectures
   - Hyperparameter tuning decisions

4. Results
   - table of all experiments
   - best and worst settings
   - confusion matrices
   - macro F1 comparison
   - discussion of why some preprocessings helped or hurt

5. Conclusion
   - main takeaways
   - limitations
   - future work

6. References
   - libraries
   - papers
   - tools
   - dataset/project PDF

## Report checklist table

In [ ]:
report_checklist = pd.DataFrame({
    "section": [
        "Abstract",
        "Introduction",
        "Dataset and EDA",
        "Preprocessing",
        "Word Representations",
        "Models",
        "Hyperparameter Tuning",
        "Results",
        "Best vs Worst Comparison",
        "Conclusion",
        "References"
    ],
    "what_to_include": [
        "overall approach, techniques, best result",
        "task motivation and dataset summary",
        "label distribution, duplicates, artifacts, word clouds, plots",
        "no/extreme/optimum preprocessing and why",
        "TF-IDF and Skip-gram implementation details",
        "ML, DNN, RNN, GRU, LSTM, bidirectional variants",
        "manual tuning choices and reasons",
        "accuracy, macro F1, confusion matrix, classification report",
        "why the best worked and why the worst failed",
        "main findings, limitations, future work",
        "IEEE-style citations for tools, libraries, papers, dataset PDF"
    ]
})
report_checklist

## Viva preparation points

Be ready to explain:
- why the train/test split was not changed
- why macro F1 matters here
- why you created exactly three preprocessing variants
- why you chose the optimum preprocessing decisions from EDA
- why a specific TF-IDF + ML model was chosen
- how Skip-gram embeddings were trained
- why one recurrent model performed better than another
- what the confusion matrix tells you about class confusion
- how class imbalance and duplicates affected the results